In [1]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.neighbors import NearestNeighbors

torch.set_num_threads(4)


In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

ALPHA_HOURS   = 3
SAMPLING_RATE = 4
ALPHA         = (ALPHA_HOURS * 3600) // SAMPLING_RATE   # 2700 samples = 3 h

SEGMENTS    = 10
NEIGHBOR_N  = 4
HIDDEN_SIZE = 64
EPOCHS      = 20
LR          = 1e-3
BATCH_SIZE  = 128   # larger than original (32) for speed; fits in T4 RAM fine

TRAIN_DATA_DIR  = "/kaggle/input/datasets/ani3hhh/phm-data/data/train"
TRAIN_FAULT_DIR = "/kaggle/input/datasets/ani3hhh/phm-data/data/train/train_faults"
TEST_DATA_DIR   = "/kaggle/input/datasets/ani3hhh/phm-data/data/test"

CHECKPOINT_DIR = "/kaggle/working/apnomo_checkpoints"
ENFORCER_DIR   = "/kaggle/working/enforcer_inputs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ENFORCER_DIR,   exist_ok=True)

PRIMARY_SIGNALS = [
    "FLOWCOOLPRESSURE",
    "FLOWCOOLFLOWRATE",
    "IONGAUGEPRESSURE",
    "ETCHBEAMCURRENT",
    "ETCHSUPPRESSORCURRENT",
    "ETCHSOURCEUSAGE",
    "ACTUALSTEPDURATION",
]

AUX_SIGNALS = [
    "ETCHBEAMVOLTAGE",
    "ETCHSUPPRESSORVOLTAGE",
    "ETCHGASCHANNEL1READBACK",
    "ETCHPBNGASREADBACK",
    "FIXTURETILTANGLE",
    "ROTATIONSPEED",
    "ACTUALROTATIONANGLE",
    "FIXTURESHUTTERPOSITION",
    "ETCHAUXSOURCETIMER",
    "ETCHAUX2SOURCETIMER",
]

ENGINEERED_COLS = [
    "FC_PRESSURE_FLOW_RATIO",
    "FC_PRESSURE_ROLLMEAN",
    "FC_PRESSURE_ROLLSTD",
]

ALL_FEATURE_COLS = PRIMARY_SIGNALS + AUX_SIGNALS + ENGINEERED_COLS   # 20
N_FEATURES       = len(ALL_FEATURE_COLS)
print(f"N_FEATURES = {N_FEATURES}")   # must be 20


Using device: cuda
N_FEATURES = 20


In [3]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    if "FLOWCOOLPRESSURE" in df.columns and "FLOWCOOLFLOWRATE" in df.columns:
        denom = df["FLOWCOOLFLOWRATE"].replace(0, np.nan)
        df["FC_PRESSURE_FLOW_RATIO"] = (df["FLOWCOOLPRESSURE"] / denom).fillna(0)
    if "FLOWCOOLPRESSURE" in df.columns:
        df["FC_PRESSURE_ROLLMEAN"] = (
            df["FLOWCOOLPRESSURE"].rolling(60, min_periods=1).mean())
        df["FC_PRESSURE_ROLLSTD"] = (
            df["FLOWCOOLPRESSURE"].rolling(60, min_periods=1).std().fillna(0))
    return df


def select_features(df: pd.DataFrame) -> pd.DataFrame:
    cols = []
    for col in ALL_FEATURE_COLS:
        if col not in df.columns:
            df[col] = 0.0
        cols.append(col)
    return df[cols]


In [4]:
class GRUExtractor(nn.Module):
    """Two-layer GRU feature extractor with dropout regularisation."""

    def __init__(self, input_dim: int):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            HIDDEN_SIZE,
            num_layers=2,
            batch_first=True,
            dropout=0.2,
        )

    def forward(self, x):
        _, h = self.gru(x)
        return h[-1]   # last-layer hidden state

print(f"GRUExtractor input_dim will be {N_FEATURES}")


GRUExtractor input_dim will be 20


In [5]:
def load_machine(file_path: str, fault_path: str | None = None):
    df = pd.read_csv(file_path)
    timestamps = df["time"].values if "time" in df.columns else np.arange(len(df))

    df = add_engineered_features(df)   # adds 3 engineered cols

    raw_primary = {
        col: df[col].values.copy() if col in df.columns else np.zeros(len(df))
        for col in PRIMARY_SIGNALS
    }

    df   = select_features(df)         # selects all 20 cols
    df   = df.ffill().bfill().fillna(0)
    data = df.values.astype(np.float32)

    labels = np.zeros(len(data), dtype=np.float32)
    if fault_path and os.path.exists(fault_path):
        fault_df    = pd.read_csv(fault_path)
        fault_times = fault_df.iloc[:, 0].values
        for ft in fault_times:
            horizon_start = ft - ALPHA * SAMPLING_RATE
            mask = (timestamps >= horizon_start) & (timestamps < ft)
            labels[mask] = 1

    return data, labels, timestamps, raw_primary


In [6]:
def create_segments(data: np.ndarray, labels: np.ndarray):
    X, y = [], []
    for i in range(0, len(data) - ALPHA, ALPHA):
        seg   = data[i: i + ALPHA]
        label = 1 if np.any(labels[i: i + ALPHA]) else 0
        X.append(seg)
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def create_raw_segments(raw_primary: dict, n_windows: int) -> list:
    segs = []
    for w in range(n_windows):
        start  = w * ALPHA
        end    = start + ALPHA
        window = {
            col: arr[start:end] if end <= len(arr) else arr[start:]
            for col, arr in raw_primary.items()
        }
        segs.append(window)
    return segs


In [7]:
def neighbor_oversample(X: np.ndarray, y: np.ndarray) -> tuple:
    X_min = X[y == 1]
    X_maj = X[y == 0]

    if len(X_min) == 0:
        return X, y

    flat        = X_min.reshape(len(X_min), -1)
    k           = min(NEIGHBOR_N, len(X_min) - 1)
    n_synthetic = len(X_maj) - len(X_min)

    if n_synthetic <= 0:
        return X, y

    if k < 1:
        idx   = np.random.choice(len(X_min), size=n_synthetic, replace=True)
        X_syn = X_min[idx]
    else:
        nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm="ball_tree").fit(flat)
        _, indices = nbrs.kneighbors(flat)
        X_syn_list = []
        for _ in range(n_synthetic):
            i   = np.random.randint(len(X_min))
            j   = indices[i, np.random.randint(1, k + 1)]
            lam = np.random.rand()
            X_syn_list.append(X_min[i] + lam * (X_min[j] - X_min[i]))
        X_syn = np.array(X_syn_list, dtype=np.float32)

    y_syn = np.ones(len(X_syn), dtype=np.float32)
    return np.concatenate([X, X_syn]), np.concatenate([y, y_syn])


In [8]:
def train_feature_extractor(files: list, scaler: StandardScaler):
    print("Fitting global scaler on training data …")
    all_data = []
    for file in files:
        base       = os.path.basename(file).replace("_DC_train.csv", "")
        fault_file = os.path.join(TRAIN_FAULT_DIR, f"{base}_train_fault_data.csv")
        data, _, _, _ = load_machine(file, fault_file)
        all_data.append(data)
    scaler.fit(np.concatenate(all_data, axis=0))
    del all_data
    print("Scaler fitted.")

    model      = GRUExtractor(N_FEATURES).to(DEVICE)
    classifier = nn.Linear(HIDDEN_SIZE, 1).to(DEVICE)
    optimizer  = torch.optim.Adam(
        list(model.parameters()) + list(classifier.parameters()), lr=LR)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion  = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([10.0], device=DEVICE))

    best_loss = float("inf")

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0

        # Stream one machine at a time — never holds >1 machine in RAM
        for file in files:
            base       = os.path.basename(file).replace("_DC_train.csv", "")
            fault_file = os.path.join(TRAIN_FAULT_DIR, f"{base}_train_fault_data.csv")

            data, labels, _, _ = load_machine(file, fault_file)
            data = scaler.transform(data)

            X, y = create_segments(data, labels)
            del data, labels
            if len(X) == 0:
                continue

            X, y = neighbor_oversample(X, y)
            perm = np.random.permutation(len(X))
            X, y = X[perm], y[perm]

            # Batch training on this machine's segments
            X_t = torch.tensor(X);  del X
            y_t = torch.tensor(y);  del y

            for i in range(0, len(X_t), BATCH_SIZE):
                bX     = X_t[i: i + BATCH_SIZE].to(DEVICE)
                by     = y_t[i: i + BATCH_SIZE].to(DEVICE)
                logits = classifier(model(bX)).squeeze(-1)
                loss   = criterion(logits, by)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += loss.item()

            del X_t, y_t
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        scheduler.step()

        tag = " ◄ best" if epoch_loss < best_loss else ""
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(),
                       os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        print(f"Epoch {epoch+1:02d}/{EPOCHS}  loss {epoch_loss:.3f}{tag}")

    model.load_state_dict(
        torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt")))
    print("Best model weights restored.")
    return model


In [9]:
def extract_features(model: GRUExtractor, files: list, scaler: StandardScaler,
                     fault_dir: str | None = None):
    model.eval()
    feats, all_labels   = [], []
    raw_segs_by_machine = []

    with torch.no_grad():
        for file in files:
            base = (os.path.basename(file)
                    .replace("_DC_train.csv", "")
                    .replace("_DC_test.csv",  ""))

            fault_file = None
            if fault_dir:
                candidate = os.path.join(fault_dir, f"{base}_train_fault_data.csv")
                if os.path.exists(candidate):
                    fault_file = candidate

            data, labels, _, raw_primary = load_machine(file, fault_file)
            data = scaler.transform(data)

            X, y = create_segments(data, labels)
            if len(X) == 0:
                raw_segs_by_machine.append([])
                continue

            raw_segs = create_raw_segments(raw_primary, n_windows=len(X))
            raw_segs_by_machine.append(raw_segs)

            # Batch inference — much faster than one segment at a time
            X_t = torch.tensor(X)
            machine_feats = []
            for i in range(0, len(X_t), BATCH_SIZE):
                out = model(X_t[i: i + BATCH_SIZE].to(DEVICE)).cpu().numpy()
                machine_feats.append(out)
            machine_feats = np.concatenate(machine_feats)

            feats.extend(machine_feats.tolist())
            all_labels.extend(y.tolist())

    return (
        np.array(feats,      dtype=np.float32),
        np.array(all_labels, dtype=np.float32),
        raw_segs_by_machine,
    )


In [10]:
def build_segment_features(full_feats: np.ndarray, full_labels: np.ndarray):
    n        = len(full_feats)
    seg_size = max(1, n // SEGMENTS)
    seg_X, seg_y = [], []
    for q in range(SEGMENTS):
        start = q * seg_size
        end   = (q + 1) * seg_size if q < SEGMENTS - 1 else n
        seg_X.append(full_feats[start:end])
        seg_y.append(full_labels[start:end])
    return seg_X, seg_y


def train_predictors(features: np.ndarray, labels: np.ndarray):
    seg_X, seg_y = build_segment_features(features, labels)
    predictors   = []
    for q in range(SEGMENTS):
        X_cum = np.concatenate(seg_X[q:], axis=0)
        y_cum = np.concatenate(seg_y[q:], axis=0)
        clf   = LogisticRegression(max_iter=500, class_weight="balanced", C=0.5)
        clf.fit(X_cum, y_cum)
        predictors.append(clf)
        print(f"Segment {q+1:02d} predictor trained  "
              f"(pos={int(y_cum.sum())}/{len(y_cum)})")
    return predictors


def compute_earliness(segs: list, y: np.ndarray) -> float:
    vals = [s / SEGMENTS for s, label in zip(segs, y)
            if label == 1 and s is not None]
    return float(np.mean(vals)) if vals else 1.0


def find_best_threshold(prob_matrix: list, y_true: np.ndarray):
    BETA      = 0.5
    all_probs = np.unique(np.array(prob_matrix).flatten())
    if len(all_probs) > 200:
        all_probs = np.percentile(all_probs, np.linspace(0, 100, 200))
    thresholds = [(all_probs[i] + all_probs[i + 1]) / 2
                  for i in range(len(all_probs) - 1)]
    if not thresholds:
        return 0.5

    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        preds, segs = [], []
        for row in prob_matrix:
            pred, seg = 0, None
            for i, p in enumerate(row):
                if p >= t:
                    pred, seg = 1, i
                    break
            preds.append(pred)
            segs.append(seg)
        f1        = f1_score(y_true, preds, zero_division=0)
        earliness = compute_earliness(segs, y_true)
        score     = BETA * f1 + (1 - BETA) * (1 - earliness)
        if score > best_score:
            best_score = score
            best_t     = t
    return best_t


In [11]:
def build_alarm_events(prob_matrix, threshold, test_labels,
                       raw_segs_by_machine, test_files):
    alarm_events = []
    flat_raw     = [seg for m in raw_segs_by_machine for seg in m]
    machine_ids  = []
    for file, machine_segs in zip(test_files, raw_segs_by_machine):
        base = (os.path.basename(file)
                .replace("_DC_test.csv",  "")
                .replace("_DC_train.csv", ""))
        machine_ids.extend([base] * len(machine_segs))

    for w_idx, (row, true_label) in enumerate(zip(prob_matrix, test_labels)):
        pred, alarm_seg = 0, None
        for seg_idx, p in enumerate(row):
            if p >= threshold:
                pred, alarm_seg = 1, seg_idx
                break
        if pred == 0:
            continue

        remaining_horizon = SEGMENTS - alarm_seg - 1
        remaining_seconds = remaining_horizon * ALPHA * SAMPLING_RATE
        alarm_events.append({
            "machine_id":        machine_ids[w_idx] if w_idx < len(machine_ids) else f"machine_{w_idx}",
            "window_idx":        w_idx,
            "alarm_segment":     alarm_seg,
            "remaining_horizon": remaining_horizon,
            "remaining_seconds": remaining_seconds,
            "prob_trace":        np.array(row),
            "true_label":        int(true_label),
            "raw_signals":       flat_raw[w_idx] if w_idx < len(flat_raw) else {},
        })
    return alarm_events


In [12]:
def main():
    all_train_files = sorted(glob.glob(os.path.join(TRAIN_DATA_DIR, "*_DC_train.csv")))
    test_files_raw  = sorted(glob.glob(os.path.join(TEST_DATA_DIR,  "*_DC_test.csv")))

    # ── The dedicated test CSVs have NO fault labels (PHM challenge format).
    # ── Their timestamps start after all known fault times → test anomalies = 0 always.
    # ── Correct approach: hold out the last 5 TRAIN machines as our eval set.
    # ── They have matching fault files and correct timestamps.
    np.random.seed(42)
    indices     = np.random.permutation(len(all_train_files))
    eval_idx    = indices[:5]
    train_idx   = indices[5:]
    train_files = [all_train_files[i] for i in sorted(train_idx)]
    eval_files  = [all_train_files[i] for i in sorted(eval_idx)]

    print(f"Train machines : {len(train_files)}")
    print(f"Eval  machines : {len(eval_files)}")
    print(f"Train files    : {[os.path.basename(f) for f in train_files]}")
    print(f"Eval  files    : {[os.path.basename(f) for f in eval_files]}")

    # ── Also keep the raw test files for the enforcer (unlabelled inference) ──
    print(f"\nUnlabelled test machines (enforcer only): {len(test_files_raw)}")

    scaler = StandardScaler()
    model  = train_feature_extractor(train_files, scaler)

    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "feature_extractor.pt"))
    joblib.dump(scaler,            os.path.join(CHECKPOINT_DIR, "scaler.pkl"))

    # ── Evaluate on held-out train machines (have fault labels) ──────────────
    train_feats, train_labels, _ = extract_features(
        model, train_files, scaler, fault_dir=TRAIN_FAULT_DIR)
    eval_feats, eval_labels, eval_raw_segs = extract_features(
        model, eval_files,  scaler, fault_dir=TRAIN_FAULT_DIR)

    print(f"\nTrain windows: {len(train_labels)}  anomaly: {int(train_labels.sum())}")
    print(f"Eval  windows: {len(eval_labels)}   anomaly: {int(eval_labels.sum())}")

    predictors = train_predictors(train_feats, train_labels)
    joblib.dump(predictors, os.path.join(CHECKPOINT_DIR, "predictors.pkl"))

    prob_matrix = [
        [clf.predict_proba(feat.reshape(1, -1))[0][1] for clf in predictors]
        for feat in eval_feats
    ]

    threshold = find_best_threshold(prob_matrix, eval_labels)
    print(f"\nBest threshold: {threshold:.6f}")

    preds, segs = [], []
    for row in prob_matrix:
        pred, seg = 0, None
        for i, p in enumerate(row):
            if p >= threshold:
                pred, seg = 1, i
                break
        preds.append(pred)
        segs.append(seg)

    precision = precision_score(eval_labels, preds, zero_division=0)
    recall    = recall_score(   eval_labels, preds, zero_division=0)
    f1        = f1_score(       eval_labels, preds, zero_division=0)
    earliness = compute_earliness(segs, eval_labels)

    print("\n========= EVAL METRICS (held-out train machines) =========")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"Earliness-T : {earliness:.4f}  (lower = earlier warning)")

    # ── Build enforcer payload from eval machines (labelled) ─────────────────
    alarm_events = build_alarm_events(
        prob_matrix=prob_matrix,
        threshold=threshold,
        test_labels=eval_labels,
        raw_segs_by_machine=eval_raw_segs,
        test_files=eval_files,
    )

    # ── Additionally run inference on the raw (unlabelled) test CSVs ─────────
    if test_files_raw:
        print(f"\nRunning inference on {len(test_files_raw)} unlabelled test machines …")
        raw_feats, raw_labels, raw_segs_test = extract_features(
            model, test_files_raw, scaler, fault_dir=None)
        raw_prob_matrix = [
            [clf.predict_proba(feat.reshape(1, -1))[0][1] for clf in predictors]
            for feat in raw_feats
        ]
        raw_alarms = build_alarm_events(
            prob_matrix=raw_prob_matrix,
            threshold=threshold,
            test_labels=raw_labels,   # all zeros — unlabelled
            raw_segs_by_machine=raw_segs_test,
            test_files=test_files_raw,
        )
        print(f"  Alarms fired on unlabelled test data: {len(raw_alarms)}")
        alarm_events = alarm_events + raw_alarms

    enforcer_payload = {
        "alarm_events":    alarm_events,
        "threshold":       threshold,
        "alpha":           ALPHA,
        "sampling_rate":   SAMPLING_RATE,
        "segments":        SEGMENTS,
        "primary_signals": PRIMARY_SIGNALS,
    }
    payload_path = os.path.join(ENFORCER_DIR, "alarm_events.pkl")
    joblib.dump(enforcer_payload, payload_path)

    print(f"\nEnforcer payload saved → {payload_path}")
    print(f"  Total alarm events   : {len(alarm_events)}")
    labelled_tp = sum(e['true_label'] for e in alarm_events)
    labelled_fp = sum(1 - e['true_label'] for e in alarm_events
                      if e['true_label'] != -1)
    print(f"  True-positive alarms : {labelled_tp}")
    print(f"  False-positive alarms: {labelled_fp}")


if __name__ == "__main__":
    main()


Train machines : 15
Eval  machines : 5
Train files    : ['02_M01_DC_train.csv', '02_M02_DC_train.csv', '03_M01_DC_train.csv', '03_M02_DC_train.csv', '04_M01_DC_train.csv', '04_M02_DC_train.csv', '05_M02_DC_train.csv', '06_M01_DC_train.csv', '06_M02_DC_train.csv', '07_M01_DC_train.csv', '07_M02_DC_train.csv', '08_M01_DC_train.csv', '09_M01_DC_train.csv', '10_M01_DC_train.csv', '10_M02_DC_train.csv']
Eval  files    : ['01_M01_DC_train.csv', '01_M02_DC_train.csv', '05_M01_DC_train.csv', '08_M02_DC_train.csv', '09_M02_DC_train.csv']

Unlabelled test machines (enforcer only): 5
Fitting global scaler on training data …
Scaler fitted.
Epoch 01/20  loss 566.920 ◄ best
Epoch 02/20  loss 440.151 ◄ best
Epoch 03/20  loss 345.584 ◄ best
Epoch 04/20  loss 306.560 ◄ best
Epoch 05/20  loss 276.427 ◄ best
Epoch 06/20  loss 297.125
Epoch 07/20  loss 290.937
Epoch 08/20  loss 263.663 ◄ best
Epoch 09/20  loss 248.664 ◄ best
Epoch 10/20  loss 236.035 ◄ best
Epoch 11/20  loss 227.403 ◄ best
Epoch 12/20  lo